In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from crewai import LLM
import yaml
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv(r'D:\vector_databse\.env')

In [ ]:
os.chdir(r'D:\vector_databse\agentic ai\crew ai')

In [ ]:
with open('agent.yaml', 'r') as f:
    agent_config = yaml.safe_load(f)

with open('task.yaml', 'r') as f:
    task_config = yaml.safe_load(f)

print('Agent Config:', agent_config)
print('Task Config:', task_config)

In [ ]:
class PolicyRetrieverTool(BaseTool):
    name: str = "Policy Retriever"
    description: str = "Retrieves relevant company policy information for customer queries."

    def _run(self, query: str) -> str:
        policies = []
        base_path = r'D:\vector_databse'
        for filename in os.listdir(base_path):
            if filename.endswith('.txt'):
                with open(os.path.join(base_path, filename), 'r') as f:
                    policies.append(f.read())
        return '\n\n'.join(policies)

class CustomerDataTool(BaseTool):
    name: str = "Customer Data Tool"
    description: str = "Retrieves customer data from the database."

    def _run(self, customer_id: str = None) -> str:
        df = pd.read_csv(r'D:\vector_databse\customers.csv')
        if customer_id:
            result = df[df['customer_id'] == customer_id]
        else:
            result = df
        return result.to_string()

In [ ]:
cfg = agent_config['customer_support_agent']

llm = LLM(model='gemini/gemini-1.5-flash', api_key=os.getenv('GOOGLE_API_KEY'))

customer_support_agent = Agent(
    role=cfg['role'].strip(),
    goal=cfg['goal'].strip(),
    backstory=cfg['backstory'].strip(),
    tools=[PolicyRetrieverTool(), CustomerDataTool()],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

print(f'Agent created: {customer_support_agent.role}')

In [ ]:
customer_query = "I took a gold loan 4 months ago and want to close it early. What are the foreclosure charges?"

support_task_cfg = task_config['support_task']

support_task = Task(
    description=support_task_cfg['description'].strip().format(customer_query=customer_query),
    expected_output=support_task_cfg['expected_output'].strip(),
    agent=customer_support_agent
)

print(f'Task created: {support_task.description[:80]}...')

In [ ]:
crew = Crew(
    agents=[customer_support_agent],
    tasks=[support_task],
    verbose=True
)

result = crew.kickoff()
print('\n=== RESULT ===')
print(result)